### Download the EuroSAT dataset
Eurosat is a dataset and deep learning benchmark for land use and land cover classification. The dataset is based on Sentinel-2 satellite images covering 13 spectral bands and consisting out of 10 classes with in total 27,000 labeled and geo-referenced images

https://github.com/phelber/eurosat

In [ ]:
!wget https://zenodo.org/records/7711810/files/EuroSAT_RGB.zip

### Unzip the dataset

In [ ]:
!unzip EuroSAT_RGB.zip

### Import packages

In [ ]:
import numpy as np
import os
from matplotlib import image as im
from matplotlib import pyplot as pl
import keras
from keras import layers
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay

### Load the images to RAM

In [ ]:
classes = os.listdir('EuroSAT_RGB')
print('Available classes', classes)
print()

class_index_dict = {c:i for i,c in enumerate(classes)}

W = 64
data_inputs = []
data_targets = []
for c_i, c in enumerate(classes):
  n = len(os.listdir(os.path.join('EuroSAT_RGB', c)))
  temp_inputs = np.empty((n, W, W, 3))
  temp_targets = np.zeros((n, len(classes)))
  for f_i, fn in enumerate(os.listdir(os.path.join('EuroSAT_RGB', c))):
    temp_inputs[f_i,:,:,:] = im.imread(os.path.join('EuroSAT_RGB', c, fn))
  print(c, temp_inputs.shape)
  data_inputs.append(temp_inputs)
  temp_targets[:,c_i] = 1
  data_targets.append(temp_targets)

data_inputs = np.vstack(data_inputs)
data_targets = np.vstack(data_targets)

print()
print('inputs', data_inputs.shape)
print('targets', data_targets.shape)

In [ ]:
# For the second experiment (see Report section below), find a way to group the classes.
# This is an example of how to do it: you should complete the ? marks
# The use of this example is not mandatory. You can write your own code

#class_groups_dict = {'WaterBodies': [?],
#                     'Urban': [?],
#                     'Vegetation': [?],
#                     'Forest': [?]}

#data_targets_4 = []
#for group_name, classes_inside in class_groups_dict.items():
#  temp = np.zeros(?)
#  for class_name in classes_inside:
#    temp = np.logical_or(?, data_targets[:,class_index_dict[class_name]])
#  data_targets_4.append(temp)
#data_targets_4 = np.vstack(data_targets_4).T.astype(float)

#print('targets', data_targets_4.shape)

### Explore the data in the dataset
This step is useful to know how to normalize the dataset. This histogram shows that the data can be normalized to the (0, 1) interval by dividing the data by 255

In [ ]:
pl.figure(figsize=(10,5))
pl.hist(data_inputs.flatten(), 100)
pl.axvline(np.max(data_inputs), c='r', linestyle='--')
pl.grid()
print('maximum value:', np.max(data_inputs))

### Create a model
The model has 10 layers:

Input image  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&#8595;  
Conv2D (32 x (3x3))  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&#8595;  
MaxPooling2D (2x2)  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&#8595;  
Conv2D (32 x (3x3))  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&#8595;  
MaxPooling2D (2x2)  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&#8595;  
Conv2D (32 x (3x3))  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&#8595;  
MaxPooling2D (2x2)  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&#8595;  
Conv2D (32 x (3x3))  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&#8595;  
MaxPooling2D (2x2)  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&#8595;  
Dense (32)  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&#8595;  
Dense (10)  

In [ ]:
model = keras.Sequential(
    [
        keras.Input(shape=(64, 64, 3)),
        layers.Conv2D(32, kernel_size=(3, 3), activation="relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Conv2D(32, kernel_size=(3, 3), activation="relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Conv2D(32, kernel_size=(3, 3), activation="relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Conv2D(32, kernel_size=(3, 3), activation="relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Flatten(),
        layers.Dropout(0.5),
        layers.Dense(32, activation='tanh'),
        layers.Dense(len(classes), activation="softmax")
    ]
)

model.summary()

model.compile(optimizer='Adam', loss='categorical_crossentropy')

### Prepare the data for training, validating and testing
This cell will split the dataset in three parts: training (80%), validation (10%) and testing (10%). The input images are normalized by dividing the values by 255

In [ ]:
n = data_inputs.shape[0]
rem_index = np.arange(n)
validation_index = np.random.choice(rem_index, int(0.1*n), replace=False)
rem_index = list(set(rem_index) - set(validation_index))
test_index = np.random.choice(rem_index, int(0.1*n), replace=False)
train_index = list(set(rem_index) - set(test_index))

validation_input = data_inputs[validation_index,:,:,:] / 255.0
validation_target = data_targets[validation_index,:]
test_input = data_inputs[test_index,:,:,:] / 255.0
test_target = data_targets[test_index,:]
train_input = data_inputs[train_index,:,:,:] / 255.0
train_target = data_targets[train_index,:]

### Fit model parameters
This cell runs the training algorithm for a given number of epochs and evaluates the loss in both the training and validation datasets

In [ ]:
history = model.fit(train_input, train_target, batch_size=64, epochs=10, validation_data=(validation_input, validation_target))

### Verify there is no overfitting
It is possible to verify that there were no overfitting by showing the training and validation losses through epochs

In [ ]:
pl.figure(figsize=(16,5))
pl.plot(history.history['loss'], label='train')
pl.plot(history.history['val_loss'], label='validation')
pl.legend()
pl.grid()

### Predict on unseen data

In [ ]:
# train_prediction = model.predict(train_input)
validation_prediction = model.predict(validation_input)
test_prediction = model.predict(test_input)

### Assess model quality


In [ ]:
from sklearn.metrics import f1_score

f1_score_validation = f1_score(np.argmax(validation_target, axis=1), np.argmax(validation_prediction, axis=1), average=None)
f1_score_test = f1_score(np.argmax(test_target, axis=1), np.argmax(test_prediction, axis=1), average=None)

def plot_bars(data, title):
  pl.bar(np.arange(len(classes)), data)
  pl.axhline(0.5, c='r', linestyle='--')
  pl.ylim(0, 1)
  pl.xticks(np.arange(len(classes)), classes, rotation=90)
  pl.title(title)
  pl.grid()

plot_bars(f1_score_validation, 'validation')
pl.show()

plot_bars(f1_score_test, 'test')
pl.show()


In [ ]:
fig, ax1 = pl.subplots(nrows=1, ncols=1, figsize=(15,15))

cm = confusion_matrix(np.argmax(test_target, axis=1), np.argmax(test_prediction, axis=1), labels=np.arange(len(classes)))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
disp.plot(ax=ax1)


## Report
1- Show the confusion matrix on the test dataset that has 10 classes. On which classes does the model produce more errors? comment briefly.

2- Show one example of image correctly classified and one example of image wrongly classified. Do it for each one of the 10 classes.

3- Join some classes to create a new output dataset containing only 4 classes:
- Water bodies (SeaLake, River)
- Urban (Industrial, Highway, Residential)
- Vegetation (Herbaceous, Pasture, PermanentCrop, AnnualCrop)
- Forest

4- Train a new model using the same methodology (roughly, the same neural network architecture and same train, validation, test split)

5- Show the confusion matrix on the test dataset with 4 classes. On which classes does the model produce more errors? Why?

6- Show one example of image correctly classified and one example of image wrongly classified. Do it for each one of the 4 classes

7- Conclusions:
- What do you think about the images giving wrong predictions? What is the problem with those images?
- Why the second experiment works better/worse than the first one?
- Talk about the behavior of the training algorithm. What would happen if the number of epochs is increased?
- How to solve the problem of unbalanced classes in the second experiment (4 classes)? Can you observe a consequence of such imbalance ?
- Do you think that the complexity of the model should be higher (e.g., more trainable parameters)? please, explain.